Data Preperation

In [ ]:
#load and indexing the data
import json, numpy as np, pandas as pd
from pycocotools import mask as mu

coco = json.load(open("data/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"))

imgs = pd.DataFrame(coco["images"])
anns = pd.DataFrame(coco["annotations"])

imgs["annotator"] = imgs["id"].str.split("-", n=1).str[0]
imgs["obs"]       = imgs["id"].str.split("-", n=1).str[1]
imgs["time"]      = pd.to_datetime(imgs["obs"].str[:14], format="%Y%m%d%H%M%S")
imgs["instrument"] = imgs["obs"].str[14:]

def ann_to_mask(ann, h=2048, w=2048):
    rle = mu.merge(mu.frPyObjects(ann["segmentation"], h, w))
    return mu.decode(rle)

def mask_to_counts(binary_mask):
    rle = mu.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")

Below is now data analysis, which is used later

TODO - use all data collected as justification later on

In [3]:
#counts images, observations, filaments, annotators and instruments (to tell you how big the dataset is)
print("images (annotation sets):", len(imgs))
print("unique observations:", imgs["obs"].nunique())
print("annotated filaments:", len(anns))
print("annotators:", imgs["annotator"].nunique())
print("\nannotators per observation:")
print(imgs.groupby("obs")["annotator"].nunique().value_counts().sort_index())
print("\ninstruments:")
print(imgs["instrument"].value_counts())
print("\ndate range:", imgs["time"].min(), "->", imgs["time"].max())
print("\nimage sizes:", imgs[["width", "height"]].drop_duplicates().values.tolist())

images (annotation sets): 1154
unique observations: 707
annotated filaments: 8199
annotators: 37

annotators per observation:
annotator
1    411
2    145
3    151
Name: count, dtype: int64

instruments:
instrument
Lh    210
Mh    205
Bh    198
Ch    195
Th    180
Uh    166
Name: count, dtype: int64

date range: 2011-01-09 10:47:34 -> 2022-08-03 07:47:52

image sizes: [[2048, 2048]]


In [4]:
#how many filaments each image has and how large they are (show the size and imbalance of what must be segmented)
per_img = anns.groupby("image_id").size()
print("filaments per image:\n", per_img.describe())
print("\nimages with zero filaments:", len(imgs) - per_img.shape[0])
print("\nfilament area (pixels):\n", anns["area"].describe())
print("\ncategory counts:\n", anns["category_id"].value_counts())

filaments per image:
 count    1154.000000
mean        7.104853
std         4.375513
min         1.000000
25%         4.000000
50%         7.000000
75%        10.000000
max        26.000000
dtype: float64

images with zero filaments: 0

filament area (pixels):
 count     8199.000000
mean      2119.834980
std       2741.721526
min          9.000000
25%        670.000000
50%       1228.000000
75%       2438.000000
max      37739.000000
Name: area, dtype: float64

category counts:
 category_id
3    3074
2    2590
1    2535
Name: count, dtype: int64


In [5]:
#divides into 5 folds (by calendar month) - saves that to splits.csv
obs = imgs.drop_duplicates("obs")[["obs", "time", "instrument"]].copy()
obs["block"] = obs["time"].dt.to_period("M").astype(str)

K = 5
rng = np.random.RandomState(42)
sizes = obs["block"].value_counts()
order = sizes.sample(frac=1, random_state=42).sort_values(ascending=False, kind="stable").index  #big blocks first
fold_load = np.zeros(K); block_fold = {}
for b in order:
    f = int(np.argmin(fold_load)); block_fold[b] = f; fold_load[f] += sizes[b]
obs["fold"] = obs["block"].map(block_fold)

def get_split(fold, gap_days=2):
    val = obs[obs.fold == fold]
    tr = obs[obs.fold != fold]
    d = np.abs(tr["time"].values[:, None] - val["time"].values[None, :]).min(axis=1)
    tr = tr[d > np.timedelta64(gap_days, "D")]
    return tr["obs"].tolist(), val["obs"].tolist()

print("observations per fold:\n", obs["fold"].value_counts().sort_index())
print("\ninstrument mix per fold:\n", pd.crosstab(obs["fold"], obs["instrument"]))
for f in range(K):
    tr, va = get_split(f)
    print(f"fold {f}: train obs {len(tr)}, val obs {len(va)}")

out = imgs[["id", "file_name", "obs", "annotator", "time", "instrument"]].merge(obs[["obs", "fold"]], on="obs")
out.to_csv("splits.csv", index=False)

observations per fold:
 fold
0    142
1    142
2    141
3    141
4    141
Name: count, dtype: int64

instrument mix per fold:
 instrument  Bh  Ch  Lh  Mh  Th  Uh
fold                              
0           20  26  20  26  27  23
1           27  29  29  15  28  14
2           20  24  26  33  18  20
3           20  22  27  27  17  28
4           31  24  25  24  20  17
fold 0: train obs 561, val obs 142
fold 1: train obs 564, val obs 142
fold 2: train obs 561, val obs 141
fold 3: train obs 561, val obs 141
fold 4: train obs 562, val obs 141


In [6]:
#check for data leakage
for f in range(5):
    tr, va = get_split(f)
    assert len(set(tr) & set(va)) == 0, "you leaked it damn"
print("no case appears in both train and val (in any fold)")

no case appears in both train and val (in any fold)


In [13]:
#this is used to speed up training
#instead of fetching labels from the data, its cashed for faster retrieval

import os
os.makedirs("cache/labels", exist_ok=True)

for iid in imgs["id"]:
    lab = np.zeros((2048, 2048), np.uint16)
    for k, r in enumerate(rles_of(iid), 1):
        lab[mu.decode(r) > 0] = k
    np.savez_compressed(f"cache/labels/{iid}.npz", lab=lab)
print("cached", len(os.listdir("cache/labels")), "label maps")

cached 1154 label maps


In [8]:
#Spec says: Implement the Panoptic quality metric locally so that you can validate your model without spending Kaggle submissions.
from scipy.optimize import linear_sum_assignment

def pq_score(gt, pr, thr=0.5):
    if len(gt) == 0 and len(pr) == 0:
        return dict(pq=1.0, sq=1.0, rq=1.0, tp=0, fp=0, fn=0, one_to_many=0, many_to_one=0)
    if len(gt) == 0 or len(pr) == 0:
        return dict(pq=0.0, sq=0.0, rq=0.0, tp=0, fp=len(pr), fn=len(gt), one_to_many=0, many_to_one=0)
    iou = np.array(mu.iou(pr, gt, [0] * len(gt)))            # (n_pred, n_gt)
    r, c = linear_sum_assignment(-iou)
    m = iou[r, c]; m = m[m > thr]
    tp = len(m); fp = len(pr) - tp; fn = len(gt) - tp
    sq = m.mean() if tp else 0.0
    rq = tp / (tp + 0.5 * fp + 0.5 * fn)
    overlap = iou > 0
    return dict(pq=sq * rq, sq=sq, rq=rq, tp=tp, fp=fp, fn=fn,
                one_to_many=int((overlap.sum(0) > 1).sum()),
                many_to_one=int((overlap.sum(1) > 1).sum()))

#toy tests
def box(x0, y0, x1, y1):
    m = np.zeros((2048, 2048), np.uint8); m[y0:y1, x0:x1] = 1
    return mu.encode(np.asfortranarray(m))

A, B = box(100, 100, 200, 200), box(400, 100, 500, 200)
print("perfect :", pq_score([A, B], [A, B])["pq"]) #expect 1
print("merged  :", pq_score([A, B], [mu.merge([A, B], intersect=False)])["pq"]) # expect 0
print("split   :", pq_score([A], [box(100, 100, 150, 200), box(150, 100, 200, 200)])["pq"]) #halves: IoU 0.5, not > 0.5, so 0
print("shifted :", pq_score([A], [box(110, 100, 210, 200)])["pq"]) #IoU about 0.82, so about 0.82

perfect : 1.0
merged  : 0.0
split   : 0.0
shifted : 0.8181818181818182


Baseline model below

In [ ]:
#Load and augment training data

#Design choices:
#You can't train on a 2048x2048 image directly, so I crop the images to 512x512 around the fillament.
#This will always train positive matches however so its a 30% chance it crops in a random place (fine tune that value later)

#Multiply effective data:
#Augment each training image: random rotation of (0, 90, 180 or 270) degrees and 50% chance of horizontal flip.
#(this might be stupid if the fillaments have to be a certain orientation)

#Also randomly change brigtness (multiply by 0.85-1.15 and then shift by maximum of 0.2)

import cv2, time, torch, torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader

FOLD = 0
tr_obs, va_obs = get_split(FOLD)
train_ids = imgs.loc[imgs["obs"].isin(tr_obs), "id"].tolist()
val_ids   = imgs.loc[imgs["obs"].isin(va_obs), "id"].tolist()
fname = dict(zip(imgs["id"], imgs["file_name"]))
print("train sets:", len(train_ids), "| val sets:", len(val_ids))

CROP = 512
def norm(img_u8):
    return (img_u8.astype(np.float32) / 255.0 - 0.45) / 0.25

class FilamentDS(Dataset):
    def __init__(self, ids, n_samples, p_fg=0.7):
        self.ids, self.n, self.p_fg = ids, n_samples, p_fg
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        rng = np.random.default_rng()
        iid = self.ids[rng.integers(len(self.ids))]
        img = cv2.imread(f"{IMG_DIR}/{fname[iid]}", cv2.IMREAD_GRAYSCALE)
        m = (np.load(f"cache/labels/{iid}.npz")["lab"] > 0).astype(np.uint8)

        if rng.random() < self.p_fg and m.any(): #a crop that contains a filament
            ys, xs = np.nonzero(m)
            k = rng.integers(len(ys))
            y0 = int(np.clip(ys[k] - rng.integers(CROP), 0, 2048 - CROP))
            x0 = int(np.clip(xs[k] - rng.integers(CROP), 0, 2048 - CROP))
        else:                                            #random crop
            y0, x0 = rng.integers(0, 2048 - CROP + 1, size=2)
        img = img[y0:y0 + CROP, x0:x0 + CROP]
        m = m[y0:y0 + CROP, x0:x0 + CROP]

        k = rng.integers(4) #augmentation
        img, m = np.rot90(img, k), np.rot90(m, k)
        if rng.random() < 0.5:
            img, m = img[:, ::-1], m[:, ::-1]
        x = norm(np.ascontiguousarray(img))
        x = x * rng.uniform(0.85, 1.15) + rng.uniform(-0.2, 0.2)# brightness and contrast change
        return (torch.from_numpy(x)[None].float(),
                torch.from_numpy(np.ascontiguousarray(m))[None].float())

In [ ]:
#Model

#Design choices
#Firstly - a U-Net encoder is used

device = "cuda" #graphics card training
model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=1, classes=1).to(device)
bce, dice = nn.BCEWithLogitsLoss(), smp.losses.DiceLoss("binary")
loss_fn = lambda logits, y: bce(logits, y) + dice(logits, y)
print("parameters (M):", sum(p.numel() for p in model.parameters()) / 1e6)